In [1]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
import google.generativeai as genai

c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=google_api_key)

In [4]:
model = genai.GenerativeModel("gemini-2.5-flash")
room_df = pd.read_csv("rooms.csv", index_col=[0])

In [5]:
model = genai.GenerativeModel("gemini-2.5-flash",
                              system_instruction=f"""
                              Bạn tên là VBot, một trợ lý AI có nhiệm vụ hỗ trợ giải đáp thông tin cho khách hàng của homestay V.
                              Vị trí của V homestay tọa lạc tại Hội An
                              Các chức năng mà bạn hỗ trợ gồm: {', '.join(room_df['name'].to_list())}.
                              Tự động nhận biết và trả lời bằng tiếng Việt hoặc tiếng Anh theo ngôn ngữ của khách.
                              Nếu không chắc, hỏi lại khách muốn dùng ngôn ngữ nào.
                              Giúp khách đặt phòng
                              Giới thiệu các loại phòng, có giá phòng, thông tin về phòng
                              Check-in / Check-out:
                              Giờ check-in tiêu chuẩn.
                              Giờ check-out tiêu chuẩn.
                              Thông tin về check-in sớm / check-out muộn (nếu có)
                              Phòng có dịch vụ free 2 chai nước và 2 gói cà phê
                              Dịch vụ mất phí: snack, nước ngọt, giặt đồ
                              Gợi ý quán ăn, cà phê, đặc sản địa phương gần khách sạn
                              Gợi ý địa điểm du lịch, tham quan, vui chơi trong bán kính gần
                              Mỗi gợi ý nên kèm:
                              Khoảng cách / thời gian di chuyển
                              Phù hợp cho ai (gia đình, cặp đôi, đi một mình…)
                              Hãy sử dụng lại lịch sử trò chuyện để đưa ra hỗ trợ có ích hơn.
                              Đối với các câu hỏi ngoài chức năng mà bạn hỗ trợ, trả lời bằng 'Tôi đang không hỗ trợ chức năng này. Xin liên hệ nhân viên homestay V để biết thêm thông tin.'
                              """
)

In [6]:
for m in genai.list_models():
  if 'embedContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview


In [10]:
model_name = "models/gemini-embedding-001"

def embed_column(title, text, price):
    full_text = f"Room name: {title}. Description: {text}. Price: {price}"
    
    return genai.embed_content(
        model=model_name,
        content=full_text,
        task_type="retrieval_document",
        title=title
    )["embedding"]

room_df['description_emb'] = room_df.apply(
    lambda row: embed_column(row['name'], row['description'], row['price']),
    axis=1
)

In [12]:
import numpy as np

def find_best_passage(query, dataframe, colname, emb_colname):
  """
  Compute the distances between the query and each document in the dataframe
  using the dot product.
  """
  query_embedding = genai.embed_content(model=model_name,
                                        content=query,
                                        task_type="retrieval_query")
  dot_products = np.dot(np.stack(dataframe[emb_colname]), query_embedding["embedding"])
  print(dot_products)
  print(dataframe['name'].to_list())
  idx = np.argmax(dot_products)
  return dataframe.iloc[idx][colname] # Return text from index with max value   

query = "Có phòng nào không?"
answer = find_best_passage(query, room_df, 'description', 'description_emb')
answer

[0.71552273 0.71315161 0.7105823  0.70142417]
['single-room', 'double-room', 'family-room', 'vip-room']


'Giường queen, 1 tivi, 1 tủ lạnh mini'